# Using CytoTable data in Pycytominer

This walkthrough shows how to load a CytoTable profile table, normalize the morphology features, and let inferred normalization safely skip nested OME-Arrow payload columns when they are present.

In [ ]:
from pathlib import Path

from pycytominer import normalize
from pycytominer.cyto_utils import load_cytotable_profiles, load_profiles

warehouse_path = Path("../tests/test_data/cytotable/examplehuman_iceberg_warehouse")
profiles_df = load_profiles(
    warehouse_path / "warehouse" / "profiles" / "joined_profiles"
)
profiles_df = load_cytotable_profiles(warehouse_path, table_name="joined_profiles")
profiles_df.shape

In [ ]:
profiles_df = profiles_df.assign(
    Metadata_treatment=[
        "control" if image_number % 2 == 0 else "drug"
        for image_number in profiles_df["Metadata_ImageNumber"]
    ]
)

normalized_df = normalize(
    profiles=profiles_df,
    features="infer",
    meta_features="infer",
    samples="Metadata_treatment == 'control'",
    method="standardize",
)

normalized_df.filter(regex="^(Metadata_|Cells_|Cytoplasm_|Nuclei_)").head()

`normalize()` returns only the inferred metadata and profile feature columns, so nested OME-Arrow image payload columns are left out of the normalized output while the numeric morphology data are processed as usual.